# 94 — SmolVLA Q10 demonstration pretraining

Pretrains the compact Q10 critic on a deterministic, task-balanced subset of the official successful LIBERO demonstrations: **8 train + 2 validation episodes per task (400 total)**. The SmolVLA policy is frozen. Every 10-action boundary is encoded once, pooled to 128 prefix tokens, and cached on Drive. Training uses the genuine EMA Bellman target.

The validation demonstrations are all successes, so `failure AUC = nan` is expected. Read Bellman/return MAE during this phase; failure discrimination is learned and evaluated after continuation on the v3 counterfactual trees. MuJoCo and the NVIDIA EGL package are not used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Use a Colab GPU runtime.'
CACHE_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_demo_q10/cache')
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_demo_q10/checkpoints')
SOURCE_ROOT = None  # selected raw parquets stay on Colab local disk, not Drive
UPDATES = 4000
MICRO_BATCH_SIZE = 64
ENCODE_BATCH_SIZE = 16
print({'gpu': torch.cuda.get_device_name(0), 'updates': UPDATES,
       'demo_split': '8 train + 2 validation per each of 40 tasks',
       'target': 'EMA Bellman Q10', 'prefix_tokens': 128})

In [ ]:
from pnp.smolvla_demo_pretraining import run_smolvla_demo_q10_pretraining

report = run_smolvla_demo_q10_pretraining(
    cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT, source_root=SOURCE_ROOT,
    updates=UPDATES, micro_batch_size=MICRO_BATCH_SIZE,
    encode_batch_size=ENCODE_BATCH_SIZE, resume=True)
report